[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/photoelasticity/photoelasticity.ipynb)

# Photoelasticity

This example computes the photoelastic fringe patterns of a loaded transparent plate with a circular hole viewed between polarizers. Pulling the plate along its length concentrates stress around the rim of the hole and makes the material birefringent, so light polarized along one principal stress direction travels slower than light polarized along the other. Between crossed linear or circular polarizers, that phase delay produces dark and bright fringes tracing contours of the stress field.

In-plane stress is modelled as a linear map from extrinsic cut planes to intrinsic traction directions built from metric-free dyads `radial * (Cut & radial)`, while solving `(Cut & stress(Cut)).eigh(Cut | Cut)` against the spatial metric gives the principal stresses and principal cuts. Light propagation normal to the plate is modelled in spacetime with complex phasor amplitudes, where the electric field is an extrinsic bivector and each polarizer or principal stress axis sweeps an intrinsic worldsheet `axis & time`. Pairing `field & (axis & time)` reads the field component along `axis`, and summing the two principal-axis projections weighted by their stress-induced phase delays forms the optical map of the stressed plate. The companion [Poincaré sphere notebook](photoelasticity_poincare.ipynb) computes the same fringes by modelling polarization states on a sphere turned by a spatial rotor.

In Cauchy tensor and Jones-calculus notation, transforming Kirsch's polar components $(\sigma_{rr}, \sigma_{\theta\theta}, \sigma_{r\theta})$ at $\theta = \operatorname{atan2}(y, x)$ into Cartesian components gives the principal angle $\alpha = \tfrac12\operatorname{atan2}(2\sigma_{xy}, \sigma_{xx} - \sigma_{yy})$ and principal stresses $\sigma_{1,2}$, and rotating the diagonal phase matrix into the laboratory frame gives $J = R(-\alpha)\,\operatorname{diag}(e^{i k C t \sigma_1}, e^{i k C t \sigma_2})\,R(\alpha)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, concatenate
from numga.algebras import STA
from examples.util.animation import save_figure, show_animation
from examples.optics.photoelasticity import render

ga = STA
# Complex coefficients hold the in-phase and quarter-period electric field amplitudes.
mv = NumpyContext(ga, dtype=np.complex128).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
Cut = ga.gatype.vector().derive.within(Algebra("x+y+"))      # Vector+
Planar = Cut.derive.dual                                     # Antivector-
Traction = Planar                                            # Antivector-
Stress = ga.gatype((Traction, Cut))                          # Traction- <- Cut+
Field = ga.gatype.bivector()                                 # Bivector+
OpticalMap = ga.gatype((Field, Field))                       # Field+ <- Field+

# Rest-frame time direction pairing to one with mv.t, and eighth/quarter turns in the plate.
time = -mv.t.dual()                                          # [] Antivector-
eighth_turn = (mv.xy * (-np.pi / 8)).exp()                   # [] Rotor+
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                  # [] Rotor+
axis_field = (Planar & time).dual()                          # [] Field+ <- Planar-
polarizer = axis_field * ((Planar & time) & Field)           # [] Field+ <- (Planar-, Planar-, Field+)

## Stress concentration around a circular hole

Pulling the plate along `x.dual()` far from the circular hole redistributes traction around the unloaded rim so that both radial and shear tractions vanish on the boundary. Summing the radial, tangential, and shear dyads forms Kirsch's stress map without invoking a metric, and solving `(Cut & stress(Cut)).eigh(Cut | Cut)` against the spatial metric yields the principal stresses and principal directions.

In [ ]:
radius = 10.0
extent = 30.0
n_pixels = 200
tension = 30.0

In [ ]:
def kirsch_stress(positions: Planar, radius: float, remote: Stress) -> Stress:
    """Kirsch's exterior solution maps `positions` (`[...] Planar-`) outside a traction-free hole of `radius` under
    `remote` load to the local stress (`[...] Traction- <- Cut+`)."""
    distance = positions.antinorm()                                               # [...] Scalar+
    radial = positions / distance                                                 # [...] Planar-
    tangent = quarter_turn >> radial                                              # [...] Planar-
    radius_ratio = (radius / distance).squared()                                  # [...] Scalar+

    radial_cut, tangent_cut = radial.dual_inverse(), tangent.dual_inverse()       # [...] Cut+, Cut+
    mean = (Cut & remote(Cut)).contract() / 2                                     # [...] Scalar+
    anisotropy = (radial_cut & remote(radial_cut)) - mean                         # [...] Scalar+
    shear = radial_cut & remote(tangent_cut)                                      # [...] Scalar+
    radial_stress = (mean * (1 - radius_ratio)
                     + anisotropy * (1 - 4 * radius_ratio + 3 * radius_ratio.squared()))  # [...] Scalar+
    tangential_stress = (mean * (1 + radius_ratio)
                         - anisotropy * (1 + 3 * radius_ratio.squared()))         # [...] Scalar+
    shear_stress = shear * (1 + 2 * radius_ratio - 3 * radius_ratio.squared())    # [...] Scalar+
    return (radial_stress * radial * (Cut & radial)
            + tangential_stress * tangent * (Cut & tangent)
            + shear_stress * (radial * (Cut & tangent) + tangent * (Cut & radial)))  # [...] Traction- <- Cut+


def principal_stresses(stress: Stress) -> tuple[Scalar, Planar]:
    """Solving `(Cut & stress(Cut)).eigh(Cut | Cut)` yields the ascending `[..., 2] Scalar+` principal stresses and
    `[..., 2] Planar-` principal directions."""
    principal, cuts = (Cut & stress(Cut)).eigh(Cut | Cut)                         # [..., 2] Scalar+, Cut+
    return principal, cuts.dual()                                                 # [..., 2] Scalar+, Planar-


centres = (np.arange(n_pixels) + 0.5) * (2 * extent / n_pixels) - extent
positions = mv.x.dual() * centres[None, :] + mv.y.dual() * centres[:, None]       # [n_pixels, n_pixels] Planar-
pull_direction = mv.x.dual()                                                      # [] Planar-
remote = tension * pull_direction * (Cut & pull_direction)                        # [] Traction- <- Cut+
stress = kirsch_stress(positions, radius, remote)                                 # [n_pixels, n_pixels] Traction- <- Cut+
principal, directions = principal_stresses(stress)                                # [n_pixels, n_pixels, 2] Scalar+, Planar-
difference = principal[..., 1] - principal[..., 0]                                # [n_pixels, n_pixels] Scalar+
save_figure(render.stress_figure(positions, difference, radius, tension), "photoelastic_stress");

In the figure, colour shows the principal stress difference in units of the applied tension across a square window spanning three hole radii from the centre of the infinite plate, rising from zero in dark blue to three in yellow at the top and bottom of the rim. [MIT's hole-in-a-plate experiment](https://ocw.mit.edu/courses/2-002-mechanics-and-materials-ii-spring-2004/09e8137971e754b1b7290a33a100bb05_lab_4_s04.pdf) gives the analytical Kirsch derivation and experimental setup.

## Optical retardance and polariscope views

A linear polarizer along `axis` extracts `field & (axis & time)` and re-emits it along `axis_field(axis)`, while the stressed plate sums both principal-direction projections weighted by their stress-induced phase delays. Because the mean stress delays both principal directions equally, transmitted intensity depends only on the principal stress difference. Crossed linear polarizers go dark wherever a principal stress direction aligns with either polarizer axis, whereas inserting crossed quarter-wave plates at an eighth turn illuminates the plate with circularly polarized light and isolates the stress-difference fringes.

In Jones-calculus notation, a retarder in its principal basis acts as $\operatorname{diag}(e^{i\phi_1}, e^{i\phi_2})$ on a two-component complex column vector and an analyser reads $|\langle a|\psi\rangle|^2$.

In [ ]:
thickness = 1.0
wavelength = 550e-6
stress_optic = 5.5e-5
polarizer_angles = np.deg2rad([0, 22.5])
delay_per_stress = 2 * np.pi * stress_optic * thickness / wavelength

In [ ]:
def retarder(stress: Stress, delay_per_stress: float) -> OpticalMap:
    """Delaying each principal axis projection by `delay_per_stress` times its principal stress gives the stressed
    plate's optical map (`[...] Field+ <- Field+`)."""
    principal, directions = principal_stresses(stress)                            # [..., 2] Scalar+, Planar-
    delays = (principal * (1j * delay_per_stress)).exp()                          # [..., 2] Scalar+
    return (delays * polarizer(directions, directions)).sum(axis=-1)              # [...] Field+ <- Field+


def intensity(field: Field, axis: Planar) -> Scalar:
    """Pairing `field` (`[...] Field+`) with the worldsheet `axis & time` and summing the squared real and
    quarter-period amplitudes gives the transmitted intensity (`[...] Scalar+`)."""
    component = field & (axis & time)                                             # [...] Scalar+
    return component.real().squared() + (-1j * component).real().squared()        # [...] Scalar+

In [ ]:
def linear_polariscope(plate: OpticalMap, axis: Planar) -> Scalar:
    """Crossed linear polarizers with entrance along `axis` (`[...] Planar-`) transmit the intensity fraction
    (`[...] Scalar+`) passed by `plate`."""
    return intensity(plate(axis_field(axis)), quarter_turn >> axis)               # [...] Scalar+


def circular_polariscope(plate: OpticalMap, axis: Planar) -> Scalar:
    """Crossed quarter-wave plates at an eighth turn to crossed linear polarizers along `axis` (`[...] Planar-`)
    transmit the circular-polariscope intensity (`[...] Scalar+`) of `plate`."""
    fast_axis = eighth_turn >> axis                                               # [...] Planar-
    slow_axis = quarter_turn >> fast_axis                                         # [...] Planar-
    entrance_plate = polarizer(fast_axis, fast_axis) + 1j * polarizer(slow_axis, slow_axis)   # [...] Field+ <- Field+
    exit_plate = polarizer(slow_axis, slow_axis) + 1j * polarizer(fast_axis, fast_axis)       # [...] Field+ <- Field+
    incident_field = entrance_plate(axis_field(axis))                             # [...] Field+
    transmitted_field = exit_plate(plate(incident_field))                         # [...] Field+
    return intensity(transmitted_field, quarter_turn >> axis)                     # [...] Scalar+


plate = retarder(stress, delay_per_stress)                                        # [n_pixels, n_pixels] Field+ <- Field+
rotors = (mv.xy * (-polarizer_angles / 2)).exp()                                  # [2] Rotor+
axes = rotors >> mv.x.dual()                                                      # [2] Planar-
linear = linear_polariscope(plate, axes[:, None, None])                           # [2, n_pixels, n_pixels] Scalar+
circular = circular_polariscope(plate, mv.x.dual())                               # [n_pixels, n_pixels] Scalar+
intensities = concatenate((linear, circular[None]))                               # [3, n_pixels, n_pixels] Scalar+
save_figure(render.polariscope_figure(positions, intensities, radius), "photoelastic_polariscopes");

From left to right, the three panels show the crossed linear polariscope aligned with the load, the crossed linear polariscope turned by 22.5 degrees, and the dark-field circular polariscope, where the dark directional brushes of the first two views disappear and leave only the stress-difference fringes.

## Loading the plate

Ramping the applied tension from zero to its peak value and back scales the principal stresses while keeping the principal directions fixed. As the phase delays grow, bright and dark fringes nucleate at the top and bottom of the hole rim and spread outward across the plate before receding during unloading.

In [ ]:
n_frames = 180
n_load_steps = n_frames // 2 + 1
duration_ms = 40
n_colors = 12
loads = (1 - np.cos(np.linspace(0, np.pi, n_load_steps))) / 2                    # [n_load_steps]

In [ ]:
principal_polarizers = polarizer(directions, directions)                          # [n_pixels, n_pixels, 2] Field+ <- Field+
loaded_plates = (((principal * (1j * delay_per_stress * load)).exp() * principal_polarizers).sum(axis=-1)
                 for load in loads)
loading_frames = (circular_polariscope(loaded_plate, mv.x.dual()) for loaded_plate in loaded_plates)
show_animation(render.polariscope_animation(positions, loading_frames, radius), "photoelastic_loading",
               duration_ms, colors=n_colors, forward_then_back=True);

In [ ]:
# checks
outside = positions.antinorm().real() >= radius
symmetry_residual = ((mv.x & stress(mv.y)) - (mv.y & stress(mv.x))).real()        # [n_pixels, n_pixels] Scalar+
assert (symmetry_residual[outside].abs() < 1e-10).all()

expected_circular = (difference.real() * (delay_per_stress / 2)).sin().squared()  # [n_pixels, n_pixels] Scalar+
assert ((circular.real()[outside] - expected_circular[outside]).abs() < 1e-10).all()
assert (intensities.real()[:, outside] >= -1e-10).all() and (intensities.real()[:, outside] <= 1 + 1e-10).all()
assert (intensities.real()[:2, outside] <= circular.real()[outside] + 1e-10).all()